# Etape 1 : Scraper les informations principales de la première page (date, promotion, évènement et lieu)

In [78]:
from bs4 import BeautifulSoup
import requests

page = requests.get("http://www.profightdb.com/cards/wwe-cards-pg1-no-2.html")
if page.status_code == 200:
    parsedPage = BeautifulSoup(page.content, 'html5lib')
    #print(parsedPage.prettify())
else:
    print(f"Erreur {page.status_code} : impossible de récupérer la page")

links = parsedPage.select("[href^='/this-day-in-history/']")
date = [l["href"].split("/")[2].replace(".html", "") for l in links]
print(date)

promo = [p.text.strip() for p in parsedPage.find_all("strong")]
print(promo)

event = [event.text.strip() for event in parsedPage.select("[href^='/cards/wwe/']")]
print(event)

locations = [l.text.strip() for l in parsedPage.select("[href^='/locations/united-states/']")]

locations_combined = [locations[i] + ", " + locations[i+1] for i in range(0, len(locations), 2)]

for loc in locations_combined:
    print(loc)




#<td class="gray">
#<a href="/locations/united-states/utah/salt-lake-city-570.html">Salt Lake City</a>, 
#<a href="/locations/united-states/utah-44.html">Utah</a>


['11-01-2025', '10-31-2025', '10-27-2025', '10-24-2025', '10-24-2025', '10-20-2025', '10-20-2025', '10-17-2025', '10-17-2025', '10-17-2025']
['WWE', 'WWE', 'WWE', 'WWE', 'WWE', 'WWE', 'WWE', 'WWE', 'WWE', 'WWE']
["Saturday Night's Main Event", 'Friday Night Smackdown', 'Monday Night Raw', 'Main Event Taping', 'Friday Night Smackdown', 'Monday Night Raw', 'Main Event Taping - Part 2', 'Main Event Taping - Part 1', 'Friday Night Smackdown', 'WWE Live - SuperShow Japan']
Salt Lake City, Utah
Salt Lake City, Utah
Anaheim, California
Tempe, Arizona
Tempe, Arizona
Sacramento, California
Sacramento, California
San Jose, California
San Jose, California


# Etape 2 : Scraper les informations principales de chaque évènement (catcheurs, résultat, thème, durée et titre en jeu)

In [83]:
from bs4 import BeautifulSoup
import requests

url = "http://www.profightdb.com/cards/wwe/saturday-night39s-main-event-57358.html"
page = requests.get(url)
soup = BeautifulSoup(page.content, "html.parser")

matches = soup.select("div.table-wrapper table tr")[1:]  # ignorer l'en-tête
match_data = []

for tr in matches:
    tds = tr.find_all("td")
    if len(tds) < 7:
        continue

    # Gagnant (premier catcheur)
    winner = tds[1].find("a")
    winner_name = winner.text.strip() if winner else ""

    # Perdants (autres catcheurs)
    losers = [w.text.strip() for w in tds[3].find_all("a")]

    # Résultat
    result_tag = tds[2].find("i")
    result = result_tag.text.strip() if result_tag else tds[2].text.strip()

    # Durée
    duration = tds[4].text.strip()

    # Type de match
    match_type = tds[5].text.strip()

    # Titre disputé
    title = tds[6].text.strip()

    match_data.append({
        "winner": winner_name,
        "losers": losers,
        "result": result,
        "duration": duration,
        "match_type": match_type,
        "title": title
    })

for m in match_data:
    print(m)




{'winner': 'Bayley', 'losers': ['Raquel Rodriguez'], 'result': 'def.', 'duration': '', 'match_type': 'dark', 'title': ''}
{'winner': 'Cody Rhodes', 'losers': ['Drew McIntyre'], 'result': 'def. (pin)', 'duration': '18:43', 'match_type': '', 'title': 'WWE Championship'}
{'winner': 'Jade Cargill', 'losers': ['Tiffany Stratton'], 'result': 'def. (pin)', 'duration': '05:32', 'match_type': '', 'title': "WWE Women's Championship (2016)(title change)"}
{'winner': 'Dominik Mysterio', 'losers': ['Penta', 'Rusev'], 'result': 'def. (pin)', 'duration': '12:26', 'match_type': 'triple-threat', 'title': 'WWE Intercontinental Championship'}
{'winner': 'CM Punk', 'losers': ['Jey Uso'], 'result': 'def. (pin)', 'duration': '20:58', 'match_type': '', 'title': 'WWE World Heavyweight Championship(title change)'}


# Etape 3 : Tout rassembler et boucler sur toutes les pages et mettre les infos sur un CSV

In [85]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import os

BASE_URL = "http://www.profightdb.com"
NUM_PAGES = 735  # nombre de pages à scraper
OUTPUT_PATH = r"C:\Users\ema\Documents\wwe\wwe_scraping.csv"

all_rows = []

for page_num in range(1, NUM_PAGES + 1):
    print(f"Scraping page {page_num}/{NUM_PAGES}...")
    url = f"{BASE_URL}/cards/wwe-cards-pg{page_num}-no-2.html"
    page = requests.get(url)
    if page.status_code != 200:
        print(f"Impossible de récupérer la page {page_num}")
        continue

    soup = BeautifulSoup(page.content, "html.parser")

    # Dates
    dates = [l["href"].split("/")[2].replace(".html", "") 
             for l in soup.select("[href^='/this-day-in-history/']")]

    # Promotions
    promos = [p.text.strip() for p in soup.find_all("strong")]

    # Événements et liens
    events = [(e.text.strip(), BASE_URL + e["href"]) 
              for e in soup.select("[href^='/cards/wwe/']")]

    # Lieux
    locations = [l.text.strip() for l in soup.select("[href^='/locations/united-states/']")]
    locations_combined = [locations[i] + ", " + locations[i+1] 
                          for i in range(0, len(locations), 2)]

    # Boucle sur chaque événement
    for i, (event_name, event_url) in enumerate(events):
        event_date = dates[i] if i < len(dates) else ""
        event_promo = promos[i] if i < len(promos) else ""
        event_location = locations_combined[i] if i < len(locations_combined) else ""

        page_event = requests.get(event_url)
        soup_event = BeautifulSoup(page_event.content, "html.parser")

        matches = soup_event.select("div.table-wrapper table tr")[1:]
        
        for tr in matches:
            tds = tr.find_all("td")
            if len(tds) < 7:
                continue

            # Gagnant
            winner_tag = tds[1].find("a")
            winner_name = winner_tag.text.strip() if winner_tag else ""

            # Perdants
            losers = [w.text.strip() for w in tds[3].find_all("a")]

            # Résultat
            result_tag = tds[2].find("i")
            result = result_tag.text.strip() if result_tag else tds[2].text.strip()

            # Durée
            duration = tds[4].text.strip()

            # Type de match
            match_type = tds[5].text.strip()

            # Titre disputé
            title = tds[6].text.strip()

            # Ajouter une ligne pour le CSV
            all_rows.append({
                "date": event_date,
                "promotion": event_promo,
                "event_name": event_name,
                "location": event_location,
                "winner": winner_name,
                "losers": ", ".join(losers),
                "result": result,
                "duration": duration,
                "match_type": match_type,
                "title": title
            })

    # Petite pause pour ne pas surcharger le serveur
    time.sleep(1)

# Créer le DataFrame
df = pd.DataFrame(all_rows)

# Vérifier que le dossier existe
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Exporter en CSV
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Scraping terminé ! Fichier CSV enregistré dans : {OUTPUT_PATH}")


Scraping page 1/735...
Scraping page 2/735...
Scraping page 3/735...
Scraping page 4/735...
Scraping page 5/735...
Scraping page 6/735...
Scraping page 7/735...
Scraping page 8/735...
Scraping page 9/735...
Scraping page 10/735...
Scraping page 11/735...
Scraping page 12/735...
Scraping page 13/735...
Scraping page 14/735...
Scraping page 15/735...
Scraping page 16/735...
Scraping page 17/735...
Scraping page 18/735...
Scraping page 19/735...
Scraping page 20/735...
Scraping page 21/735...
Scraping page 22/735...
Scraping page 23/735...
Scraping page 24/735...
Scraping page 25/735...
Scraping page 26/735...
Scraping page 27/735...
Scraping page 28/735...
Scraping page 29/735...
Scraping page 30/735...
Scraping page 31/735...
Scraping page 32/735...
Scraping page 33/735...
Scraping page 34/735...
Impossible de récupérer la page 34
Scraping page 35/735...
Scraping page 36/735...
Scraping page 37/735...
Scraping page 38/735...
Scraping page 39/735...
Impossible de récupérer la page 39
Scr

# Etape 4 : Nettoyage

In [4]:
import pandas as pd
df = pd.read_csv(r"C:\Users\ema\Documents\WWE - scraping et analyse\wwe_scraping.csv", index_col=False)
df.to_csv('wwe_scraping.csv', index=False)
df.head()

,date,promotion,event_name,location,winner,losers,result,duration,match_type,title
0,11-01-2025,WWE,Saturday Night's Main Event,"Salt Lake City, Utah",Bayley,Raquel Rodriguez,def.,NaN,dark,NaN
1,11-01-2025,WWE,Saturday Night's Main Event,"Salt Lake City, Utah",Cody Rhodes,Drew McIntyre,def. (pin),18:43,NaN,WWE Championship
2,11-01-2025,WWE,Saturday Night's Main Event,"Salt Lake City, Utah",Jade Cargill,Tiffany Stratton,def. (pin),05:32,NaN,WWE Women's Championship (2016)(title change)
3,11-01-2025,WWE,Saturday Night's Main Event,"Salt Lake City, Utah",Dominik Mysterio,"Penta, Rusev",def. (pin),12:26,triple-threat,WWE Intercontinental Championship
4,11-01-2025,WWE,Saturday Night's Main Event,"Salt Lake City, Utah",CM Punk,Jey Uso,def. (pin),20:58,NaN,WWE World Heavyweight Championship(title change)


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40195 entries, 0 to 40194
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        40195 non-null  object
 1   promotion   40195 non-null  object
 2   event_name  40195 non-null  object
 3   location    33662 non-null  object
 4   winner      40190 non-null  object
 5   losers      40190 non-null  object
 6   result      40194 non-null  object
 7   duration    16444 non-null  object
 8   match_type  9968 non-null   object
 9   title       7273 non-null   object
dtypes: object(10)
memory usage: 3.1+ MB


In [6]:
df[["city", "state" ]] = df["location"].str.split(",", n=1, expand=True)
df.drop(columns=["location"], inplace=True)
df.head(10)

,date,promotion,event_name,winner,losers,result,duration,match_type,title,city,state
0,11-01-2025,WWE,Saturday Night's Main Event,Bayley,Raquel Rodriguez,def.,NaN,dark,NaN,Salt Lake City,Utah
1,11-01-2025,WWE,Saturday Night's Main Event,Cody Rhodes,Drew McIntyre,def. (pin),18:43,NaN,WWE Championship,Salt Lake City,Utah
2,11-01-2025,WWE,Saturday Night's Main Event,Jade Cargill,Tiffany Stratton,def. (pin),05:32,NaN,WWE Women's Championship (2016)(title change),Salt Lake City,Utah
3,11-01-2025,WWE,Saturday Night's Main Event,Dominik Mysterio,"Penta, Rusev",def. (pin),12:26,triple-threat,WWE Intercontinental Championship,Salt Lake City,Utah
4,11-01-2025,WWE,Saturday Night's Main Event,CM Punk,Jey Uso,def. (pin),20:58,NaN,WWE World Heavyweight Championship(title change),Salt Lake City,Utah
5,10-31-2025,WWE,Friday Night Smackdown,Ilja Dragunov,Nathan Frazer,def. (pin),19:05,NaN,WWE United States Championship,Salt Lake City,Utah
6,10-31-2025,WWE,Friday Night Smackdown,Carmelo Hayes,Kit Wilson,def. (pin),03:01,NaN,NaN,Salt Lake City,Utah
7,10-31-2025,WWE,Friday Night Smackdown,Alexa Bliss,Nia Jax,def. (pin),09:46,NaN,NaN,Salt Lake City,Utah
8,10-31-2025,WWE,Friday Night Smackdown,JC Mateo,"Alex Shelley, Chris Sabin",def. (pin),08:54,NaN,NaN,Salt Lake City,Utah
9,10-27-2025,WWE,Monday Night Raw,Penta,Rusev,draw (NC),09:56,NaN,NaN,Anaheim,California


In [7]:
df = df[["date", "promotion", "city", "state", "event_name", "winner", "losers", "result", "duration", "match_type", "title"]]

In [8]:
df["date"] = pd.to_datetime(df["date"], format="%m-%d-%Y")
df['duration'] = pd.to_timedelta('00:' + df['duration'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40195 entries, 0 to 40194
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   date        40195 non-null  datetime64[ns] 
 1   promotion   40195 non-null  object         
 2   city        33662 non-null  object         
 3   state       33662 non-null  object         
 4   event_name  40195 non-null  object         
 5   winner      40190 non-null  object         
 6   losers      40190 non-null  object         
 7   result      40194 non-null  object         
 8   duration    16444 non-null  timedelta64[ns]
 9   match_type  9968 non-null   object         
 10  title       7273 non-null   object         
dtypes: datetime64[ns](1), object(9), timedelta64[ns](1)
memory usage: 3.4+ MB


In [9]:
df.head()

,date,promotion,city,state,event_name,winner,losers,result,duration,match_type,title
0,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Bayley,Raquel Rodriguez,def.,NaT,dark,NaN
1,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Cody Rhodes,Drew McIntyre,def. (pin),0 days 00:18:43,NaN,WWE Championship
2,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Jade Cargill,Tiffany Stratton,def. (pin),0 days 00:05:32,NaN,WWE Women's Championship (2016)(title change)
3,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Dominik Mysterio,"Penta, Rusev",def. (pin),0 days 00:12:26,triple-threat,WWE Intercontinental Championship
4,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,CM Punk,Jey Uso,def. (pin),0 days 00:20:58,NaN,WWE World Heavyweight Championship(title change)


In [10]:
df['state_count'] = df['state'].map(df['state'].value_counts(dropna=False)).fillna(0).astype(int)
df.head()

,date,promotion,city,state,event_name,winner,losers,result,duration,match_type,title,state_count
0,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Bayley,Raquel Rodriguez,def.,NaT,dark,NaN,169
1,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Cody Rhodes,Drew McIntyre,def. (pin),0 days 00:18:43,NaN,WWE Championship,169
2,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Jade Cargill,Tiffany Stratton,def. (pin),0 days 00:05:32,NaN,WWE Women's Championship (2016)(title change),169
3,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Dominik Mysterio,"Penta, Rusev",def. (pin),0 days 00:12:26,triple-threat,WWE Intercontinental Championship,169
4,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,CM Punk,Jey Uso,def. (pin),0 days 00:20:58,NaN,WWE World Heavyweight Championship(title change),169


In [11]:
df['winner_count'] = df['winner'].map(df['winner'].value_counts(dropna=False)).fillna(0)
top20_winners = df.drop_duplicates(subset=['winner']).sort_values(by='winner_count', ascending=False).head(20)
df.head()

,date,promotion,city,state,event_name,winner,losers,result,duration,match_type,title,state_count,winner_count
0,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Bayley,Raquel Rodriguez,def.,NaT,dark,NaN,169,266
1,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Cody Rhodes,Drew McIntyre,def. (pin),0 days 00:18:43,NaN,WWE Championship,169,515
2,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Jade Cargill,Tiffany Stratton,def. (pin),0 days 00:05:32,NaN,WWE Women's Championship (2016)(title change),169,14
3,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,Dominik Mysterio,"Penta, Rusev",def. (pin),0 days 00:12:26,triple-threat,WWE Intercontinental Championship,169,103
4,2025-11-01,WWE,Salt Lake City,Utah,Saturday Night's Main Event,CM Punk,Jey Uso,def. (pin),0 days 00:20:58,NaN,WWE World Heavyweight Championship(title change),169,497


In [12]:
df['state_clean'] = df['state'].astype(str).str.strip().str.lower()

# Filtrer les states valides et créer une copie
df_valid = df[(df['state_clean'] != 'unknown') & (df['state_clean'] != 'nan')].copy()

# Convertir duration en minutes
df_valid['duration_min'] = df_valid['duration'].dt.total_seconds() / 60

# Durée moyenne par État
state_duration = df_valid.groupby('state_clean')['duration_min'].mean().sort_values(ascending=False).reset_index()



In [13]:
df['year'] = df['date'].dt.year

# Compter le nombre de matchs par année
matches_per_year = df.groupby('year').size().reset_index(name='num_matches')

In [14]:
df['loser_count'] = df['losers'].map(df['losers'].value_counts())
top20_losers = df.drop_duplicates(subset=['losers']).sort_values(by='loser_count', ascending=False).head(20)


In [15]:
performance_type = (
    df.groupby(['match_type', 'winner'])
      .size()
      .reset_index(name='wins')
      .sort_values(by='wins', ascending=False)
)

top10_winners = (
    performance_type.groupby('winner')['wins']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top10_data = performance_type[performance_type['winner'].isin(top10_winners)]

In [16]:
df['title'].value_counts().head(15)

title
WWE United States Championship          930
WWE Intercontinental Championship       908
WWE Tag Team Championship               633
WWE Championship                        511
World Heavyweight Championship (WWE)    436
WWE Divas Championship                  371
WWE World Heavyweight Championship      258
WWE Raw Women's Title                   221
WWE Raw Tag Team Championship           199
WWE Smackdown Women's Title             198
WWE Cruiserweight Title                 196
WWE Smackdown Tag Team Championship     178
WWE 24/7 Title(title change)            153
WWE World Championship                  133
World Tag Team Titles (WWE)             119
Name: count, dtype: int64

In [17]:
df_clean = df.copy()

df_clean['title'] = df_clean['title'].astype(str).str.strip()
df_clean['winner'] = df_clean['winner'].astype(str).str.strip()

titles_15 = [
    "WWE United States Championship",
    "WWE Intercontinental Championship",
    "WWE Tag Team Championship",
    "WWE Championship",
    "World Heavyweight Championship (WWE)",
    "WWE Divas Championship",
    "WWE World Heavyweight Championship",
    "WWE Raw Women's Title",
    "WWE Raw Tag Team Championship",
    "WWE Smackdown Women's Title",
    "WWE Cruiserweight Title",
    "WWE Smackdown Tag Team Championship",
    "WWE 24/7 Title(title change)",
    "WWE World Championship",
    "World Tag Team Titles (WWE)"
]

# Filtrer uniquement ces titres
df_titles = df_clean[df_clean['title'].isin(titles_15)].copy()

title_counts = df_titles['title'].value_counts().reset_index()
title_counts.columns = ['title', 'num_matches']

top_winners = []

for t in titles_15:
    df_title = df_titles[df_titles['title'] == t]
    if not df_title.empty:
        winner_counts = df_title['winner'].value_counts()
        top_winner = winner_counts.idxmax()
        wins = winner_counts.max()
        top_winners.append({'title': t, 'winner': top_winner, 'wins': wins})

top_winners_df = pd.DataFrame(top_winners)

In [18]:
# Top 10 catcheurs toutes années
top10_wrestlers = df_clean['winner'].value_counts().head(10).index
df_top10 = df_clean[df_clean['winner'].isin(top10_wrestlers)]

# Tableau année x catcheur
heatmap_data = df_top10.groupby(['winner','year']).size().unstack(fill_value=0)

In [19]:
df_clean = df.copy()

# Extraire l'année
df_clean['year'] = pd.to_datetime(df_clean['date']).dt.year

top10_wrestlers = df_clean['winner'].value_counts().head(10).index
df_top10 = df_clean[df_clean['winner'].isin(top10_wrestlers)]

top10_global = (
    df_top10.groupby('winner')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='num_matches')
)


In [20]:
# Compter le nombre de chaque type de finish
finish_counts = df['result'].value_counts().reset_index()
finish_counts.columns = ['finish', 'count']

# Garder les 5 types de finish les plus fréquents
top5_finish = finish_counts.sort_values(by='count', ascending=False).head(5)


In [21]:
df['year'] = pd.to_datetime(df['date']).dt.year

# Compter le nombre de matchs par titre et par année
matches_per_year_title = df.groupby(['year','title']).size().reset_index(name='num_matches')

# Sélectionner les 5 titres avec le plus de matchs au total
top5_titles = matches_per_year_title.groupby('title')['num_matches'].sum().sort_values(ascending=False).head(5).index

# Filtrer le DataFrame pour ne garder que ces 5 titres
matches_top5 = matches_per_year_title[matches_per_year_title['title'].isin(top5_titles)]

In [22]:
df['year'] = df['date'].dt.year

# Filtrer entre 2002 et 2010 inclus
df_filtered = df[(df['year'] >= 2002) & (df['year'] <= 2010)]

# Compter le nombre de matchs par catcheur (winner)
matches_per_wrestler = df_filtered['winner'].value_counts().reset_index()
matches_per_wrestler.columns = ['wrestler', 'num_matches']

# Garder les 10 catcheurs les plus actifs
top10_wrestlers = matches_per_wrestler.head(10)


In [23]:
match_type_year = df.groupby(['year','match_type']).size().reset_index(name='num_matches')
top5_types = match_type_year.groupby('match_type')['num_matches'].sum().sort_values(ascending=False).head(5).index
match_type_year_top5 = match_type_year[match_type_year['match_type'].isin(top5_types)]


In [24]:
cena_wins = df[df['winner'].str.contains("John Cena", case=False, na=False)]

# Compter le nombre de victoires par année
cena_wins_by_year = cena_wins.groupby('year').size().reset_index(name='wins')


In [39]:
edge_wins = df[df['winner'].str.contains("Edge", case=False, na=False)]

# Compter le nombre de victoires par année
edge_wins_by_year = edge_wins.groupby('year').size().reset_index(name='wins')

In [42]:
undertaker_wins = df[df['winner'].str.contains("The Undertaker", case=False, na=False)]

# Compter le nombre de victoires par année
undertaker_wins_by_year = undertaker_wins.groupby('year').size().reset_index(name='wins')

In [25]:
import plotly.express as px
fig = px.bar(df.dropna(subset=['state']).drop_duplicates(subset=['state']), x='state', y="state_count", title='Nombre de matchs par État', color_discrete_sequence=['#FF69B4'], labels={'state':'État', 'state_count':"Nombre de matchs"})
fig.show()


In [26]:

fig = px.bar(top20_winners, x="winner", y="winner_count", title='Top winners', color_discrete_sequence=['#FF69B4'], labels={'winner':'Gagnant', 'winner_count':'Nombre de gagnant'})
fig.show()

In [27]:
fig = px.bar(top20_losers, x="losers", y="loser_count", title='Top losers', color_discrete_sequence=['#FF69B4'], labels={'loser':'Perdant', 'loser_count':'Nombre de perdants'})
fig.show()

In [28]:
fig = px.bar(state_duration, x="state_clean", y="duration_min", title='Durée moyenne des matchs par État', color_discrete_sequence=['#FF69B4'], labels={'state_clean':'États', 'duration_min':'Durée moyenne des matchs'})
fig.show()


In [29]:
fig = px.line(matches_per_year, x="year", y="num_matches", text="num_matches", title="Nombre de matchs par année", color_discrete_sequence=['#FF69B4'],
    labels={'year':'Année', 'num_matches':'Nombre de matchs'})

fig.show()

In [30]:
fig1 = px.bar(title_counts, x='title', y='num_matches', text='num_matches', title=' Nombre de matchs par titre WWE (Top 15)', color='title', color_discrete_sequence=px.colors.qualitative.Pastel
)
fig1.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    showlegend=False
)
fig1.show()


In [31]:

fig2 = px.bar(top_winners_df, x='title', y='wins', color='title', text='winner', title=' Top gagnant pour chaque titre WWE (Top 15)', labels={'title': 'Titre WWE', 'wins': 'Nombre de victoires'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig2.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    showlegend=False
)
fig2.show()

In [32]:
fig = px.imshow(
    heatmap_data,
    labels=dict(x="Année", y="Catcheur", color="Nombre de matchs"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale='pinkyl'
)
fig.update_layout(title='Activité des 10 catcheurs les plus actifs par année')
fig.show()


In [33]:
fig_bar = px.bar(
    top10_global,
    x='winner',
    y='num_matches',
    text='num_matches',
    title='Top 10 catcheurs les plus actifs (toutes années)',
    color='winner',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_bar.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    plot_bgcolor='white'
)
fig_bar.show()


In [34]:
fig = px.bar(
    top5_finish,
    x='finish',
    y='count',
    text='count',
    title='Top 5 types de finish les plus fréquents',
    color='finish',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    labels={
        'def. (pin)': 'Victoire par tombé',
        'def.': 'Victoire générale',
        'def. (sub)': 'Victoire par soumission',
        'def. (qd)': 'Victoire par disqualification / count-out',
        'draw': 'Match nul'
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    showlegend=False
)

fig.show()



In [35]:
fig = px.line(
    matches_top5,
    x='year',
    y='num_matches',
    color='title',
    title='Nombre de matchs par année pour les 5 titres les plus populaires'
)

fig.update_layout(plot_bgcolor='white')
fig.show()



In [36]:
fig = px.bar(
    top10_wrestlers,
    x='wrestler',
    y='num_matches',
    text='num_matches',
    title='Top 10 catcheurs les plus actifs (2002-2010)',
    color='num_matches',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    showlegend=False
)

fig.show()


In [37]:

px.line(match_type_year_top5, x='year', y='num_matches', color='match_type',
        title='Évolution des 5 types de match les plus fréquents')


In [40]:
fig = px.line(
    cena_wins_by_year,
    x='year',
    y='wins',
    markers=True,
    title='Évolution du nombre de victoires de John Cena par année',
    labels={'wins': 'Nombre de victoires', 'year': 'Année'},
    line_shape='spline'
)

fig.show()

In [41]:
fig = px.line(
    edge_wins_by_year,
    x='year',
    y='wins',
    markers=True,
    title='Évolution du nombre de victoires de Edge par année',
    labels={'wins': 'Nombre de victoires', 'year': 'Année'},
    line_shape='spline'
)

fig.show()

In [43]:
fig = px.line(
    undertaker_wins_by_year,
    x='year',
    y='wins',
    markers=True,
    title='Évolution du nombre de victoires de Edge par année',
    labels={'wins': 'Nombre de victoires', 'year': 'Année'},
    line_shape='spline'
)

fig.show()